In [1]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.linear_model import Ridge
from sklearn.base import BaseEstimator, RegressorMixin, clone
import pickle
from pathlib import Path
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import TimeSeriesSplit
from sklearn.ensemble import StackingRegressor
from sklearn.model_selection import GridSearchCV

In [ ]:
pkl_path = Path("data\processed_data\combined_latest_snapshot_20251215.pkl")

with open(pkl_path, "rb") as f:
    data = pickle.load(f)
    
sorted_data = sorted(data, key=lambda x: x["date"])

<>:1: SyntaxWarning: invalid escape sequence '\p'
<>:1: SyntaxWarning: invalid escape sequence '\p'
C:\Users\Geovanka\AppData\Local\Temp\ipykernel_14300\774744323.py:1: SyntaxWarning: invalid escape sequence '\p'
  pkl_path = Path("data\processed_data\combined_latest_snapshot_20251215.pkl")


In [3]:
import warnings
warnings.filterwarnings(
    "ignore",
    message="This Pipeline instance is not fitted yet"
)

In [ ]:
valid_indices = []

for t, item in enumerate(sorted_data):
    if item.get("env") is None:
        continue
    if all(grid is not None for grid in item["env"].values()):
        valid_indices.append(t)

max_valid = max(valid_indices)
print("Last valid index:", max_valid)

data = sorted_data[:max_valid + 1]

Last valid index: 78


In [5]:
def build_dataframe(data):
    rows = []

    for item in data:
        date = item["date"]
        sic = item["sic_image"]
        env = item["env"]

        H, W = sic.shape

        for i in range(H):
            for j in range(W):

                sic_val = sic[i, j]
                if np.isnan(sic_val):
                    continue

                row = {
                    "date": date,
                    "i": i,
                    "j": j,
                    "sic": sic_val
                }

                for var, grid in env.items():
                    val = grid[i, j]
                    if val is None or (isinstance(val, float) and np.isnan(val)):
                        row[var] = np.nan
                    else:
                        row[var] = val

                rows.append(row)

    return pd.DataFrame(rows)


In [6]:
df = build_dataframe(data)
print(df.shape)

(10759168, 13)


In [ ]:
for t, item in enumerate(data):
    if item.get("sic_image") is None:
        print("sic_image is None at index", t)

In [8]:
SIC_SCALE = np.nanmax(df["sic"])

df["sic"] = df["sic"] / SIC_SCALE

In [9]:
print(df.shape)
print(df.columns)

(10759168, 13)
Index(['date', 'i', 'j', 'sic', 'u10', 'v10', 't2m', 'msl', 'sst', 'tp', 'mwd',
       'mwp', 'swh'],
      dtype='object')


In [11]:
def add_lags(df, columns, lags):
    df_lagged = df.copy()
    for col in columns:
        for lag in lags:
            df_lagged[f"{col}_lag{lag}"] = df[col].shift(lag)
    return df_lagged

In [12]:
env_vars = ["u10", "v10", "t2m", "msl", "sst", "tp", "mwd", "mwp", "swh"]

df = add_lags(df, env_vars, lags=[1])

In [13]:
df = df.sort_values(["i", "j", "date"])

df["sic_lag1"] = df.groupby(["i", "j"])["sic"].shift(1)
df["sic_lag2"] = df.groupby(["i", "j"])["sic"].shift(2)

df = df.dropna().reset_index(drop=True)

print("Final samples:", len(df))
df.head(100)

Final samples: 1147288


,date,i,j,sic,u10,v10,t2m,msl,sst,tp,...,v10_lag1,t2m_lag1,msl_lag1,sst_lag1,tp_lag1,mwd_lag1,mwp_lag1,swh_lag1,sic_lag1,sic_lag2
0,2025-09-19 00:00:00+07:00,34,178,1.0,-10.657503,0.465414,285.644684,99399.812500,285.757233,0.025708,...,0.046225,285.462463,99385.375000,285.404633,0.021712,123.304077,7.034733,2.180705,1.0,1.0
1,2025-09-20 00:00:00+07:00,34,178,1.0,-6.318811,-0.759336,285.293030,98599.625000,285.598297,0.018174,...,-1.221372,285.132538,98596.351562,285.280670,0.017537,158.021591,7.470798,1.816924,1.0,1.0
2,2025-09-21 00:00:00+07:00,34,178,1.0,-6.940454,2.006952,284.922974,99003.906250,285.513031,0.020459,...,2.032830,284.837769,98984.664062,285.240631,0.021541,184.303101,8.455075,1.732342,1.0,1.0
3,2025-09-22 00:00:00+07:00,34,178,1.0,-0.845187,0.554127,283.928131,99132.882812,285.861603,0.009196,...,0.712289,283.934479,99133.570312,285.667023,0.004959,203.700974,7.781989,1.499159,1.0,1.0
4,2025-09-23 00:00:00+07:00,34,178,1.0,-1.853202,3.655674,285.075226,100134.851562,285.525177,0.006038,...,3.863356,285.010773,100100.460938,285.586700,0.007989,198.678879,6.835272,1.267874,1.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,2025-10-07 00:00:00+07:00,34,179,1.0,-0.503311,-1.218418,282.539307,101104.710938,284.623993,0.010339,...,-1.362909,282.904633,101112.789062,285.009155,0.006996,215.530441,8.919237,2.130580,1.0,1.0
96,2025-10-08 00:00:00+07:00,34,179,1.0,0.425235,0.447091,283.914459,102001.835938,284.447662,0.000063,...,1.092680,284.296967,101993.773438,284.856689,0.000212,219.808411,8.248609,1.184889,1.0,1.0
97,2025-10-09 00:00:00+07:00,34,179,1.0,-0.647408,1.553150,284.279938,102064.062500,284.431274,0.001496,...,2.313282,284.563202,102046.476562,284.815063,0.001551,221.153442,8.575982,1.596586,1.0,1.0
98,2025-10-10 00:00:00+07:00,34,179,1.0,6.010366,-3.334803,283.314850,101756.312500,284.466949,0.002017,...,-3.972539,283.714203,101796.835938,284.827057,0.002842,246.054245,6.846683,1.598286,1.0,1.0


In [14]:
# chronological split
dates = df["date"].sort_values().unique()
split_date = dates[int(len(dates) * 0.8)]

df_train = df[df["date"] <= split_date].reset_index(drop=True)
df_test  = df[df["date"] > split_date].reset_index(drop=True)

print("Train period:", df_train["date"].min(), "→", df_train["date"].max())
print("Test period :", df_test["date"].min(),  "→", df_test["date"].max())

Train period: 2025-09-19 00:00:00+07:00 → 2025-11-19 00:00:00+07:00
Test period : 2025-11-20 00:00:00+07:00 → 2025-12-04 00:00:00+07:00


In [15]:
def get_xy(df, features):
    X = df[features].values
    y = df["sic"].values
    return X, y

In [ ]:
# Ridge-ELM
class RidgeELM(Ridge):
    """
    Ridge-regularized Extreme Learning Machine
    implemented as a subclass of sklearn Ridge
    """
    
    def __init__(
        self,
        n_hidden=300,
        activation="tanh",  # tanh is more numerically stable
        alpha=1.0,
        random_state=42
    ):
        super().__init__(alpha=alpha)
        self.n_hidden = n_hidden
        self.activation = activation
        self.random_state = random_state

    def _activation(self, X):
        if self.activation == "sigmoid":
            return 1 / (1 + np.exp(-np.clip(X, -50, 50))) 
        elif self.activation == "tanh":
            return np.tanh(X)
        elif self.activation == "relu":
            return np.maximum(0, X)
        else:
            raise ValueError("Unsupported activation")

    def fit(self, X, y):
        self.scaler_ = StandardScaler()
        Xs = self.scaler_.fit_transform(X)
        
        rng = np.random.RandomState(self.random_state)

        self.W = rng.normal(size=(Xs.shape[1], self.n_hidden))
        self.b = rng.normal(size=self.n_hidden)

        H = self._activation(Xs @ self.W + self.b)
        self.ridge = Ridge(alpha=self.alpha)
        self.ridge.fit(H, y)
        
        return super().fit(H, y)

    def predict(self, X):
        Xs = self.scaler_.transform(X)
        H = self._activation(Xs @ self.W + self.b)
        return super().predict(H)

In [17]:
class SubsampledSVR(BaseEstimator, RegressorMixin):
    def __init__(
        self,
        kernel='rbf',
        C=10,
        gamma='scale',
        epsilon=0.01,
        max_samples=20000,
        random_state=42,
    ):
        self.kernel = kernel
        self.C = C
        self.gamma = gamma
        self.epsilon = epsilon
        self.max_samples = max_samples
        self.random_state = random_state
        
    def __sklearn_tags__(self):
        tags = super().__sklearn_tags__()
        tags.estimator_type = "regressor"
        return tags
        
    def fit(self, X, y):
        if len(X) > self.max_samples:
            rng = np.random.RandomState(self.random_state)
            idx = rng.choice(len(X), self.max_samples, replace=False)
            X = X[idx]
            y = y[idx]

        # Fit the actual SVR
        self.scaler_ = StandardScaler()
        X_scaled = self.scaler_.fit_transform(X)
        
        self.svr = SVR(
            kernel=self.kernel,
            C=self.C,
            gamma=self.gamma,
            epsilon=self.epsilon,
        )
        self.svr.fit(X_scaled, y)
        return self
    
    def predict(self, X):
        X_scaled = self.scaler_.transform(X)
        return self.svr.predict(X_scaled)

In [ ]:
base_models = {
    "RF": RandomForestRegressor(
        n_estimators= 200,
        max_depth= 20,
        min_samples_split=10,
        min_samples_leaf= 5,
        max_features="sqrt",
        n_jobs=-1,
        random_state=42
    ),
    "SVR": SubsampledSVR(
        kernel="rbf",
        C= 10, 
        gamma='scale', 
        epsilon= 0.01, 
        max_samples=20000
    ),
    "R_ELM": RidgeELM(
        n_hidden= 300,
        alpha= 1.0
        ) 
}

# Time-aware cross-validation 
tscv = TimeSeriesSplit(n_splits=5)

In [19]:
def subsample(X, y, max_samples=20000, random_state=42):
    if len(X) <= max_samples:
        return X, y
    rng = np.random.RandomState(random_state)
    idx = rng.choice(len(X), max_samples, replace=False)
    return X[idx], y[idx]

In [21]:
def evaluate_model(model, X, y, cv, subsample_for_svr=False):
    rmses = []

    for train_idx, test_idx in cv.split(X):
        Xtr, Xte = X[train_idx], X[test_idx]
        ytr, yte = y[train_idx], y[test_idx]
        
        if subsample_for_svr:
            # print("Subsample SVR")
            Xtr, ytr = subsample(Xtr, ytr)

        m = clone(model)
        m.fit(Xtr, ytr)
        ypred = m.predict(Xte)

        rmse = np.sqrt(mean_squared_error(yte, ypred))
        rmses.append(rmse)
        

    return np.mean(rmses)


In [ ]:
def run_feature_set_experiments(
    FEATURE_SETS,
    models,
    df_train,
    cv
):
    all_results = {}

    for fs_name, features in FEATURE_SETS.items():
        print("\n" + "=" * 70)
        print(f"FEATURE SET: {fs_name}")
        print(f"Features ({len(features)}): {features}")
        print("=" * 70)

        X_train, y_train = get_xy(df_train, features)

        fs_results = {}

        for model_name, model in models.items():
            rmse = evaluate_model(
                model,
                X_train,
                y_train,
                cv,
                subsample_for_svr=(model_name == "SVR")
            )

            fs_results[model_name] = rmse
            print(f"{model_name:10s} | CV RMSE = {rmse:.4f}")

        best_model = min(fs_results, key=fs_results.get)
        print(f"\nBest for {fs_name}: {best_model} "
              f"(RMSE = {fs_results[best_model]:.4f})")

        all_results[fs_name] = fs_results

    return all_results


In [25]:
FEATURE_SETS = {
    # =========================
    # BASELINE MEMORY
    # =========================
    "lag1_only": [
        "sic_lag1", "sic_lag2",
        "u10_lag1", "v10_lag1", "t2m_lag1", "msl_lag1",
        "sst_lag1", "tp_lag1", "mwd_lag1", "mwp_lag1", "swh_lag1"
    ],

    # =========================
    # MEMORY + INSTANTANEOUS
    # =========================
    "current_lag1": [
        "sic_lag1", "sic_lag2",
        "u10", "v10", "t2m", "msl", "sst", "tp", "mwd", "mwp", "swh",
        "u10_lag1", "v10_lag1", "t2m_lag1", "msl_lag1",
        "sst_lag1", "tp_lag1", "mwd_lag1", "mwp_lag1", "swh_lag1"
    ],

    # =========================
    # LINEAR DOMAIN
    # =========================
    "linear_lag1": [
        "sic_lag1", "sic_lag2",
        "tp_lag1", "mwd_lag1", "mwp_lag1", "swh_lag1"
    ],

    "linear_current_lag1": [
        "sic_lag1", "sic_lag2",
        "tp", "mwd", "mwp", "swh",
        "tp_lag1", "mwd_lag1", "mwp_lag1", "swh_lag1"
    ],

    # =========================
    # NONLINEAR DOMAIN
    # =========================
    "nonlinear_lag1": [
        "sic_lag1", "sic_lag2",
        "u10_lag1", "v10_lag1", "t2m_lag1", "msl_lag1", "sst_lag1"
    ],

    "nonlinear_current_lag1": [
        "sic_lag1", "sic_lag2",
        "u10", "v10", "t2m", "msl", "sst",
        "u10_lag1", "v10_lag1", "t2m_lag1", "msl_lag1", "sst_lag1"
    ]
}


In [27]:
# all the base learners predict the same thing (strong multicollinearity)
# klo pake linear regression biasa: unstable coef, small noise -> huge weight swing, poor generalization
# ridge bisa mengatasi hal ini: handles multicollinearity, fast, stable
# ridge belajar untuk tiap weight nya, klo voting kan tiap model equal weight nya

stacking_model = StackingRegressor(
    estimators=[
        ("RF", base_models["RF"]),
        ("SVR", base_models["SVR"]),
        ("R_ELM", base_models["R_ELM"])    
    ],
    final_estimator=Ridge(alpha=10),
    cv=3,
    n_jobs=-1
)

models = {
    "RF": base_models["RF"],
    "SVR": base_models["SVR"],
    "R_ELM": base_models["R_ELM"],
    "STACK": stacking_model
}


In [30]:
all_results = run_feature_set_experiments(
    FEATURE_SETS=FEATURE_SETS,
    models=models,
    df_train=df_train,
    cv=tscv
)


FEATURE SET: lag1_only
Features (11): ['sic_lag1', 'sic_lag2', 'u10_lag1', 'v10_lag1', 't2m_lag1', 'msl_lag1', 'sst_lag1', 'tp_lag1', 'mwd_lag1', 'mwp_lag1', 'swh_lag1']
RF         | CV RMSE = 0.0181
SVR        | CV RMSE = 0.0297
R_ELM      | CV RMSE = 0.0747
STACK      | CV RMSE = 0.0171

Best for lag1_only: STACK (RMSE = 0.0171)

FEATURE SET: current_lag1
Features (20): ['sic_lag1', 'sic_lag2', 'u10', 'v10', 't2m', 'msl', 'sst', 'tp', 'mwd', 'mwp', 'swh', 'u10_lag1', 'v10_lag1', 't2m_lag1', 'msl_lag1', 'sst_lag1', 'tp_lag1', 'mwd_lag1', 'mwp_lag1', 'swh_lag1']
RF         | CV RMSE = 0.0208
SVR        | CV RMSE = 0.0338
R_ELM      | CV RMSE = 0.0903
STACK      | CV RMSE = 0.0204

Best for current_lag1: STACK (RMSE = 0.0204)

FEATURE SET: linear_lag1
Features (6): ['sic_lag1', 'sic_lag2', 'tp_lag1', 'mwd_lag1', 'mwp_lag1', 'swh_lag1']
RF         | CV RMSE = 0.0156
SVR        | CV RMSE = 0.0179
R_ELM      | CV RMSE = 0.0204
STACK      | CV RMSE = 0.0130

Best for linear_lag1: STACK (RM

In [31]:
best_combo = None
best_rmse = np.inf

for fs_name, res in all_results.items():
    for model_name, rmse in res.items():
        if rmse < best_rmse:
            best_rmse = rmse
            best_combo = (fs_name, model_name)

print("GLOBAL BEST")
print(f"Feature set: {best_combo[0]}")
print(f"Model:       {best_combo[1]}")
print(f"RMSE:        {best_rmse:.4f}")


GLOBAL BEST
Feature set: linear_lag1
Model:       STACK
RMSE:        0.0130
